# Simulación del juego 4 en raya

## Introducción

En este notebook exploraremos diferentes estrategias para simular la toma de decisiones en el juego 4 en raya. 4 en raya es un juego de estrategia para dos jugadores que consiste en alinear cuatro fichas del mismo color en un tablero vertical.

Reglas:
 - El juego clásico se juega en un tablero de 7 columnas por 6 filas (aunque en esta implementación pueden tomar valores entre 4 y 10).
 - Los jugadores se turnan para dejar caer una ficha en una de las columnas.
 - La ficha cae hasta ocupar la posición más baja disponible en esa columna.
 - El objetivo es alinear cuatro fichas consecutivas del mismo color (en línea horizontal, vertical o diagonal).
 - Gana el primer jugador que consiga alinear cuatro fichas.

 En este notebook exploraremos diferentes estrategias para simular la toma de decisiones en 4 en raya.

In [6]:
import sys
import os
import pandas as pd
import random
import numpy as np

# Fijar la semilla
SEED = 123
random.seed(SEED)
np.random.seed(SEED)

# Agregar los directorios al path para poder importar los módulos
sys.path.append(os.path.abspath("../games"))
sys.path.append(os.path.abspath("../methods"))
sys.path.append(os.path.abspath("../selection"))
sys.path.append(os.path.abspath("../graphviz"))

# Importar la clase Connect4Game y las funciones de los métodos
from connect4 import Connect4Game
from monte_carlo import MCPlay, MCAgent
from monte_carlo_tree_search import MCTS, MCTSPlay, Node, MCTSAgent
from epsilon_greedy import EpsilonGreedy
from softmax import Softmax
from adaptive_softmax import AdaptiveSoftmax
from ucb1 import UCB1
from ucb2 import UCB2
from gradiente_preferencias import GradienteDePreferencias
from MCTS_graphviz import generate_tree_MCTS

## Implementación de la clase `Connect4Game`

En esta sección, se explica cómo ha sido implementada la clase `Connect4Game` que modela el juego. Para ello, a continuación se presenta una breve descripción de cada uno de los métodos que la componen:

 - `__init__(self, rows=6, cols=7, starting_player=1)`: Se utiliza para inicializar el juego. Inicializa el tablero de tamaño `rows` x `cols` con casillas vacías (0). Sus parámetros incluyen:
   - `rows`: número de filas del tablero (de 4 a 10).
   - `cols`: número de columnas del tablero (de 4 a 10).
   - `starting_player`: el jugador que empieza el juego (1 o -1).

 - `valid_actions(self)`: Este método devuelve las acciones válidas que un jugador puede realizar, es decir, las columnas donde se pueden colocar fichas porque no están llenas.

 - `action(self, col)`: Este método simula un turno del juego, encontrando la primera fila vacía en la columna seleccionada y colocando la ficha del jugador en esa posición, y devuelve el nuevo estado del juego. Además, realiza una comprobación para asegurarse de que la acción solicitada sea válida.

 - `terminal(self)`: Verifica si el juego ha terminado, esto es, si un jugador ha alineado 4 fichas consecutivas, o si el tablero está lleno (lo que resulta en un empate).

 - `winner(self)`: Devuelve quién es el ganador. Si el juego ha terminado, devuelve el número del jugador ganador (1 o -1) o un 0 si ha habido empate, pero si no ha terminado devuelve 0.

 - `draw(self)`: Este método dibuja el estado actual del juego, imprimiendo el tablero en formato de matriz, donde cada celda representa una ficha (1, 2, o 0 si está vacía).

## Simulaciones

En esta sección, se realizan simulaciones con distintas estrategias para la toma de decisiones en el juego. Comenzamos creando una instancia del juego `Connect4Game`, y visualizamos su estado inicial. Por defecto, se juega con un tablero con 6 filas y 7 columnas. Sin embargo, estos parámetros se pueden modificar con `rows` y `cols`.

In [2]:
# Crear una instancia del juego 4 en raya con los valores predeterminados
game = Connect4Game()

# Mostrar el estado inicial del juego
game.draw()

[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]


A continuación, se presentan los distintos métodos utilizados para realizar las simulaciones.

## Monte-Carlo

En primer lugar, se implementa el método de Monte-Carlo para estimar las mejores jugadas. 

In [ ]:
# Crear una instancia del juego 4 en raya
game = Connect4Game(rows=6, cols=7, starting_player=1)

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 1000

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = int(input(f"¿En qué columna quieres colocar tu ficha (0-{game.cols-1})?"))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue  # Volver a pedir jugada

    else:  # Turno de la IA (Monte-Carlo)
        print("\nTurno de la computadora...")
        game = MCPlay(game, num_simulations)

    game.draw()  # Mostrar el estado después del turno

# Anunciar el ganador
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
elif game.winner() == -1:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")
else:
    print("\n¡Es un empate!")

[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]

Tu turno...
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 1, 0, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
[0, 0, 0, 1, 0, 0, 0]

Tu turno...
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
[0, 0, 0, 1, 1, 0, 0]

Turno de la computadora...
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
[0, 0, 2, 1, 1, 0, 0]

Tu turno...
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
[0, 0, 2, 1, 1, 1, 0]

Turno de la computadora...
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 

### Monte-Carlo Tree Search

En esta sección, se utiliza el método de Monte-Carlo Tree Search para estimar las mejores jugadas en el juego 4 en raya. Este algoritmo está compuesto por distintas fases, que son:

 1. **Selección:** En esta fase, se parte del nodo raíz y se desciende por el árbol seleccionando nodos hijos sucesivamente según una estrategia. Esto continúa hasta llegar a un nodo hoja, es decir, un nodo que tiene al menos un hijo potencial al que no se le ha aplicado ninguna simulación todavía. Entre las estrategias de selección encontramos: $\epsilon$-greedy, Softmax, Adaptive Softmax, UCB1, UCB2 y Gradiente de Preferencias.

 2. **Expansión:** A partir del nodo hoja seleccionado, se genera un nuevo nodo hijo, es decir, se aplica un movimiento válido que aún no ha sido explorado desde el nodo hoja.

 3. **Simulación:** Desde el nodo hijo recién creado, se completa una jugada aleatoria hasta alcanzar un estado terminal (por ejemplo, una victoria, derrota o empate). Mediante esta simulación, se puede estimar el resultado potencial de seguir esa línea de decisión.

 4. **Retropropagación:** Los resultados de la simulación se propagan hacia atrás, usándose para actualizar las estadísticas de los nodos, que son recorridos desde el nodo expandido hasta el nodo raíz. Esto permite que en futuras decisiones se refuercen las rutas más prometedoras y se descarten las menos efectivas.

In [ ]:
# Crear una instancia del juego 4 en raya
game = Connect4Game()

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 1000

# Escoger el método de selección
selection_algorithm_class = GradienteDePreferencias

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = int(input(f"¿En qué columna quieres colocar tu ficha (0-{game.cols-1})?"))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue

    else:  # Turno de la IA con MCTS
        print("\nTurno de la computadora...")
        game = MCTSPlay(game, num_simulations, selection_algorithm_class)

    game.draw()

# Resultado
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
elif game.winner() == -1:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")
else:
    print("\n¡Es un empate!")

A continuación, se utiliza `graphviz` como herramienta de depuración, generando un diagrama con el árbol de posiciones siguientes a partir de una determinada posición inicial, dada por un nodo raíz. Cada nodo del árbol representa un estado del juego, el cual se alcanza mediante una secuencia de movimientos, y contiene información como el turno del jugador, el número de visitas y la recompensa de cada jugador. Las aristas indican las acciones tomadas para pasar de un estado al siguiente. Se puede especificar la profundidad máxima del árbol generado.

In [ ]:
# Crear estado inicial del juego
initial_state = Connect4Game(rows=6, cols=7, starting_player=1)

# Visualizar el árbol de MCTS
root_node = Node(initial_state, None)
mcts = MCTS(root_node, UCB1, simulations=1000, c=1)
mcts.run()
id_to_node = generate_tree_MCTS(root_node, max_depth=1, filename="depuration_trees/arbol_connect4", format="jpg")

Los nodos en el árbol aparecen identificados mediante un ID. Para poder visualizar cuál es el estado de cada nodo, la función `generate_tree_MCTS` devuelve un diccionario que asocia cada ID al nodo correspondiente.

In [28]:
node = id_to_node['nodo4']

node.state.draw()

[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 1, 0, 0, 0]


## Simulaciones comparativas

In [24]:
def play_comparative_game(agent1, agent2, rows=5, cols=5, starting_player=1):
    game = Connect4Game(rows=rows, cols=cols, starting_player=starting_player)
    agents = {1: agent1, -1: agent2}

    while not game.terminal():
        game = agents[game.turn].move(game)

    return game.winner()

In [25]:
def run_simulations(agent_1, agent_2, rows=5, cols=5, n_games=100):
    results = []

    for i in range(n_games):
        # Alternar jugador inicial
        starting_player = 1 if i % 2 == 0 else -1

        # Asignar agentes según quién empieza
        if starting_player == 1:
            winner = play_comparative_game(agent_1, agent_2, rows, cols, starting_player)
        else:
            winner = play_comparative_game(agent_2, agent_1, rows, cols, starting_player)
            # Invertir perspectiva
            winner *= -1

        results.append(winner)

    df = pd.DataFrame(results, columns=["winner"])
    win_rate = (df["winner"] == 1).mean()
    print(f"% de partidas ganadas por el agente 1: {win_rate:.2%} ({df['winner'].value_counts().to_dict()})")
    return df

In [26]:
agent_mc = MCAgent(300)
agent_mcts_eps = MCTSAgent(300, EpsilonGreedy, epsilon = 0.2)
agent_mcts_UCB1 = MCTSAgent(300, UCB1, c=1)
agent_mcts_UCB2 = MCTSAgent(300, UCB2, alpha=0.5)
agent_mcts_soft = MCTSAgent(300, Softmax, tau = 1)
agent_mcts_adapsoft = MCTSAgent(300, AdaptiveSoftmax, tau_0 = 1, alpha = 0.5)
agent_mcts_grad = MCTSAgent(300, GradienteDePreferencias, alpha = 0.2)

### Monte Carlo vs MCTS con $\epsilon$-greedy

In [28]:
df_mc_mcts_eps = run_simulations(agent_mc, agent_mcts_eps, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 57.00% ({1: 57, -1: 41, 0: 2})


### Monte Carlo vs MCTS con UCB1

In [30]:
df_mc_mcts_UCB1 = run_simulations(agent_mc, agent_mcts_UCB1, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 52.00% ({1: 52, -1: 47, 0: 1})


### Monte Carlo vs MCTS con UCB2

In [31]:
df_mc_mcts_UCB2 = run_simulations(agent_mc, agent_mcts_UCB2, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 96.00% ({1: 96, -1: 4})


### Monte Carlo vs MCTS con Softmax

In [32]:
df_mc_mcts_soft = run_simulations(agent_mc, agent_mcts_soft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 72.00% ({1: 72, -1: 25, 0: 3})


### Monte Carlo vs MCTS con Softmax Adaptativo

In [33]:
df_mc_mcts_adapsoft = run_simulations(agent_mc, agent_mcts_adapsoft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 76.00% ({1: 76, -1: 21, 0: 3})


### Monte Carlo vs MCTS con Gradiente de Preferencias

In [ ]:
df_mc_mcts_grad = run_simulations(agent_mc, agent_mcts_grad, rows=5, cols=5, n_games=100)

### MCTS con $\epsilon$-greedy vs MCTS con UCB1

In [34]:
df_mcts_eps_UCB1 = run_simulations(agent_mcts_eps, agent_mcts_UCB1, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 39.00% ({-1: 56, 1: 39, 0: 5})


### MCTS con $\epsilon$-greedy vs MCTS con UCB2

In [35]:
df_mcts_eps_UCB2 = run_simulations(agent_mcts_eps, agent_mcts_UCB2, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 92.00% ({1: 92, -1: 7, 0: 1})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax

In [36]:
df_mcts_eps_soft = run_simulations(agent_mcts_eps, agent_mcts_soft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 68.00% ({1: 68, -1: 30, 0: 2})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax Adaptativo

In [37]:
df_mcts_eps_adapsoft = run_simulations(agent_mcts_eps, agent_mcts_adapsoft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 70.00% ({1: 70, -1: 26, 0: 4})


### MCTS con $\epsilon$-greedy vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_eps_grad = run_simulations(agent_mcts_eps, agent_mcts_grad, rows=5, cols=5, n_games=100)

### MCTS con UCB1 vs MCTS con UCB2

In [38]:
df_mcts_UCB1_UCB2 = run_simulations(agent_mcts_UCB1, agent_mcts_UCB2, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 96.00% ({1: 96, -1: 3, 0: 1})


### MCTS con UCB1 vs MCTS con Softmax

In [39]:
df_mcts_UCB1_soft = run_simulations(agent_mcts_UCB1, agent_mcts_soft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 74.00% ({1: 74, -1: 19, 0: 7})


### MCTS con UCB1 vs MCTS con Softmax Adaptativo

In [40]:
df_mcts_UCB1_adapsoft = run_simulations(agent_mcts_UCB1, agent_mcts_adapsoft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 75.00% ({1: 75, -1: 16, 0: 9})


### MCTS con UCB1 vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB1_grad = run_simulations(agent_mcts_UCB1, agent_mcts_grad, rows=5, cols=5, n_games=100)

### MCTS con UCB2 vs MCTS con Softmax

In [41]:
df_mcts_UCB2_soft = run_simulations(agent_mcts_UCB2, agent_mcts_soft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 26.00% ({-1: 72, 1: 26, 0: 2})


### MCTS con UCB2 vs MCTS con Softmax Adaptativo

In [42]:
df_mcts_UCB2_adapsoft = run_simulations(agent_mcts_UCB2, agent_mcts_adapsoft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 22.00% ({-1: 76, 1: 22, 0: 2})


### MCTS con UCB2 vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_UCB2, agent_mcts_grad, rows=5, cols=5, n_games=100)

### MCTS con Softmax vs MCTS con Softmax Adaptativo

In [43]:
df_mcts_soft_adapsoft = run_simulations(agent_mcts_soft, agent_mcts_adapsoft, rows=5, cols=5, n_games=100)

% de partidas ganadas por el agente 1: 53.00% ({1: 53, -1: 40, 0: 7})


### MCTS con Softmax vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_soft, agent_mcts_grad, rows=5, cols=5, n_games=100)